In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import json
from datasets import Dataset
from tqdm import tqdm
import os
import time
import evaluate

# -----------------------
# CONFIG
# -----------------------
OUTPUT_ROOT = "Cross_Validation_2_FineTune_155k"
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
batch_size = 64  # adjust depending on memory

# -----------------------
# Load CV summary and pick best fold
# -----------------------
summary_path = os.path.join(OUTPUT_ROOT, "cv_summary.json")
with open(summary_path, "r", encoding="utf-8") as f:
    fold_summaries = json.load(f)

best_fold = max(fold_summaries, key=lambda x: x["metrics"].get("eval_bleu", 0))["fold"]
print(f"✅ Best fold selected: Fold {best_fold}")

fold_dir = os.path.join(OUTPUT_ROOT, f"fold_{best_fold}")
tokenizer = AutoTokenizer.from_pretrained(fold_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(fold_dir)
model.eval()

translator = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    device=-1  # CPU; use 0 if GPU available
)

# -----------------------
# Function to evaluate a dataset file
# -----------------------
def evaluate_dataset(file_path, output_dir, separator="++++$++++"):
    os.makedirs(output_dir, exist_ok=True)
    en_sentences, te_sentences = [], []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Detect if separator exists
    if separator in lines[0]:
        for line in lines:
            if separator in line:
                en, te = line.strip().split(separator)
                en_sentences.append(en)
                te_sentences.append(te)
    else:
        if len(lines) % 2 != 0:
            lines = lines[:-1]
        for i in range(0, len(lines), 2):
            en_sentences.append(lines[i].strip())
            te_sentences.append(lines[i+1].strip())

    dataset = Dataset.from_dict({"en": en_sentences, "te": te_sentences})

    preds, refs = [], []
    total_batches = len(dataset) // batch_size + 1
    start_time = time.time()

    for i in tqdm(range(0, len(dataset), batch_size), total=total_batches, desc=f"Translating {os.path.basename(file_path)}"):
        batch = dataset[i:i+batch_size]["en"]
        outputs = translator(batch, max_length=MAX_TARGET_LENGTH)
        preds.extend([o["translation_text"] for o in outputs])
        refs.extend([[r] for r in dataset[i:i+batch_size]["te"]])

    end_time = time.time()
    print(f"✅ Translation completed in {(end_time - start_time)/60:.2f} minutes")

    # Compute metrics
    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")
    bleu_score = bleu.compute(predictions=preds, references=refs)
    chrf_score = chrf.compute(predictions=preds, references=refs)
    metrics = {"bleu": bleu_score["score"], "chrF": chrf_score["score"]}

    # Save predictions, references, metrics
    with open(os.path.join(output_dir, "preds.txt"), "w", encoding="utf-8") as pf:
        pf.writelines([p+"\n" for p in preds])
    with open(os.path.join(output_dir, "refs.txt"), "w", encoding="utf-8") as rf:
        rf.writelines([r[0]+"\n" for r in refs])
    with open(os.path.join(output_dir, "metrics.json"), "w", encoding="utf-8") as mf:
        json.dump(metrics, mf, indent=2)

    print(f"\n📊 Evaluation Results ({os.path.basename(file_path)}):", metrics)


# -----------------------
# List your test files here
# -----------------------
test_files = [
    "500_data.txt",    # new data
    "700_data.txt"     # trained data
]

# Output folders for each
for file_path in test_files:
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    output_dir = os.path.join("score_check", f"{base_name}_results")
    evaluate_dataset(file_path, output_dir)

✅ Best fold selected: Fold 1


Device set to use cpu
Translating 500_data.txt: 100%|██████████████████████████████████████████████████████████| 8/8 [11:34<00:00, 86.78s/it]


✅ Translation completed in 11.57 minutes



📊 Evaluation Results (500_data.txt): {'bleu': 3.778679478766941, 'chrF': 28.018798844901333}


Translating 700_data.txt: 100%|████████████████████████████████████████████████████████| 11/11 [10:10<00:00, 55.49s/it]


✅ Translation completed in 10.17 minutes

📊 Evaluation Results (700_data.txt): {'bleu': 70.36423281958915, 'chrF': 85.8607400612283}
